# Extended Figure 5 — Class-Summed Gene Expression: Combination vs. Single Agents

## Overview
Generates a barplot of class-summed Δlog₁(x+1) expression (mean ± SEM) for anabolic and catabolic gene panels, comparing a combination treatment and its two single agents against an IL-1β baseline.  
This notebook produces the Extended Figure associated with Figure 5 (drug combination synergy and biological validation).

## Inputs
- `../Rdata/pub/mac_cmb0.1.sct.h5ad` — AnnData object, combination treatments at 0.1 μM
- `../Rdata/pub/mac_sin0.1.sct.h5ad` — AnnData object, single-drug treatments at 0.1 μM
  - Both objects require `adata.obs['drug_name']` with condition labels including `'inflammatory'` as baseline

## Outputs
- `output/main/Figure5ext_classsum_barplot.pdf`

## Python environment
Tested with Python 3.9. Required packages: `scanpy`, `numpy`, `matplotlib`, `scipy`.  
Run the final cell to record exact package versions used.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import scanpy as sc
import scipy.stats

# Paths — relative to this notebook's location (03_downstream_analysis/)
DATA_DIR = Path("../Rdata/pub")
OUT_DIR  = Path("output/main")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
mac_cmb01 = sc.read_h5ad(DATA_DIR / "mac_cmb0.1.sct.h5ad")
mac_sin01  = sc.read_h5ad(DATA_DIR / "mac_sin0.1.sct.h5ad")

In [ ]:
# Gene panels — must match R/config.R definitions exactly
GENES_ANABOLIC    = ["Acan", "Sox9", "Col2a1", "Matn1", "Matn3", "Ucma",
                     "Ccnd3", "Gadd45g", "Pth1r", "Gm26633", "Col27a1"]
GENES_CATABOLIC   = ["Mmp3", "Mmp13", "Il6", "Il17b", "Adamts5", "Igfbp3",
                     "Ccl2", "Cxcl5", "Cxcl1", "Fosl2", "Tlr2", "Tnfrsf1b"]
GENES_HOUSEKEEPING = ["Hprt", "Actb", "Gapdh", "B2m", "Ubc", "Ppia", "Rpl23"]

# Baseline condition label (IL-1β treated)
COND_DISEASE = "inflammatory"

In [ ]:
def sem_of_class_sum(adata, pert, genes, control_label=COND_DISEASE):
    ctrl_X = adata[adata.obs["drug_name"] == control_label][:, list(genes)].X
    pert_X = adata[adata.obs["drug_name"] == pert][:, list(genes)].X

    try: ctrl_X = ctrl_X.toarray()
    except Exception: pass
    try: pert_X = pert_X.toarray()
    except Exception: pass

    ctrl_X = np.log1p(ctrl_X)
    pert_X = np.log1p(pert_X)

    # baseline-correct per gene, then sum across genes per cell
    corrected   = pert_X - ctrl_X.mean(axis=0, keepdims=True)
    class_score = corrected.sum(axis=1)

    m  = np.mean(class_score)
    se = scipy.stats.sem(class_score)
    return m, se

In [6]:
# Select perturbations
combo = "SB431542&SANT-1"
x_g = 'SB431542'
y_g = 'SANT-1'


In [ ]:
from collections import OrderedDict

classes = OrderedDict([
    ("Anabolic",  GENES_ANABOLIC),
    ("Catabolic", GENES_CATABOLIC),
])

fig, ax = plt.subplots(figsize=(7, 4.5))
width = 0.25

xticks      = []
xticklabels = []

for i, (cls_name, genes) in enumerate(classes.items()):
    m_x, se_x = sem_of_class_sum(mac_sin01, x_g,   genes)
    m_y, se_y = sem_of_class_sum(mac_sin01, y_g,   genes)
    m_c, se_c = sem_of_class_sum(mac_cmb01, combo, genes)

    base = i * 3.0
    ax.bar(base - width, m_x, width=width, yerr=se_x, edgecolor="black",
           linewidth=1, capsize=3, label=f"Single: {x_g}"  if i == 0 else None)
    ax.bar(base,         m_y, width=width, yerr=se_y, edgecolor="black",
           linewidth=1, capsize=3, label=f"Single: {y_g}"  if i == 0 else None)
    ax.bar(base + width, m_c, width=width, yerr=se_c, edgecolor="black",
           linewidth=1, capsize=3, color="firebrick",
           label="Combination" if i == 0 else None)

    xticks.append(base)
    xticklabels.append(cls_name)

ax.set_xticks(xticks)
ax.set_xticklabels(xticklabels)
ax.axhline(y=0, linewidth=1, color="black")
ax.set_ylabel("Summed Δlog₁(x+1) expression (mean ± SEM)")
ax.set_title(f"Class-summed expression changes: {combo}")
ax.legend(ncol=3, loc="lower left")

out_pdf = OUT_DIR / "Figure5ext_classsum_barplot.pdf"
plt.savefig(out_pdf, format="pdf", bbox_inches="tight")
print("Saved:", out_pdf)

In [ ]:
# Environment reproducibility — run once after executing the notebook to record versions
import subprocess, sys, datetime

result = subprocess.run([sys.executable, "-m", "pip", "freeze"],
                        capture_output=True, text=True)
env_path = OUT_DIR.parent.parent / "session_info" / "figure4d_pip_freeze.txt"
env_path.parent.mkdir(parents=True, exist_ok=True)
with open(env_path, "w") as f:
    f.write(f"# figure4d.ipynb — pip freeze\n")
    f.write(f"# Recorded: {datetime.datetime.now().isoformat()}\n\n")
    f.write(result.stdout)
print(f"Environment recorded to: {env_path}")